In [2]:
import numpy as np
import pandas as pd
from collections import defaultdict

class StandardNaiveBayes:
    def __init__(self):
        self.prior_probabilities = {}
        self.conditional_probabilities = defaultdict(lambda: defaultdict(dict))
        self.classes = []
        self.features = []

    def fit(self, df, feature_columns, target_column):
        self.features = feature_columns
        self.classes = df[target_column].unique()
        total_samples = len(df)

        print("\n" + "="*80)
        print("📐 STANDARD NAIVE BAYES MATH BREAKDOWN (NO LAPLACE SMOOTHING)")
        print("="*80)

        # 1. Calculate Prior Probabilities P(C_i)
        print("\n[Step 1: Prior Probabilities P(C)]")
        for c in self.classes:
            class_count = len(df[df[target_column] == c])
            self.prior_probabilities[c] = class_count / total_samples
            print(f"  • P(class = '{c}') = {class_count} / {total_samples} = {self.prior_probabilities[c]:.4f}")

        # 2. Calculate Raw Conditional Probabilities P(X_j | C_i)
        print("\n[Step 2: Raw Conditional Probabilities P(Feature = Value | Class)]")
        for feature in self.features:
            unique_values = df[feature].unique()
            print(f"\n  📊 Feature Column: '{feature}'")
            
            for c in self.classes:
                class_df = df[df[target_column] == c]
                total_class_count = len(class_df)
                
                for val in unique_values:
                    matching_count = len(class_df[class_df[feature] == val])
                    
                    # RAW CALCULATION: No +1.0 in the numerator, no unique value multiplier in the denominator
                    raw_prob = matching_count / total_class_count
                    self.conditional_probabilities[feature][c][val] = raw_prob
                    
                    print(f"    - P({feature}='{val}' | class='{c}') = {matching_count} / {total_class_count} = {raw_prob:.4f}")

    def predict_single(self, sample_dict):
        """Calculates posterior probabilities using raw multiplications"""
        posteriors = {}
        math_steps = []

        for c in self.classes:
            probability = self.prior_probabilities[c]
            calculation_str = f"{probability:.4f}"
            
            for feature, val in sample_dict.items():
                if feature in self.features:
                    # Fetches raw probability; defaults to 0.0 if value is completely unseen in training
                    cond_prob = self.conditional_probabilities[feature][c].get(val, 0.0)
                    probability *= cond_prob
                    calculation_str += f" * {cond_prob:.4f}"
            
            posteriors[c] = probability
            math_steps.append(f"  • P(Class='{c}' | X) ∝ {calculation_str} = {probability:.6f}")
            
        winning_class = max(posteriors, key=posteriors.get)
        return winning_class, posteriors, math_steps


# ==========================================
# Execution Main Program
# ==========================================
if __name__ == "__main__":
    # Load dataset from the file
    df = pd.read_csv('Buys_Computer_Dataset.csv')

    features = ["Age", "Income", "Student", "Credit_rating"]
    target = "class:buys_comp"

    # Initialize and train the raw classifier
    nb_classifier = StandardNaiveBayes()
    nb_classifier.fit(df, features, target)

    print("\n" + "="*80)
    print("🔮 PREDICTING A NEW TEST SAMPLE (RAW CALCULATIONS)")
    print("="*80)

    # You can change the test samples here as needed:
    test_sample = {
        "Age": "Y",
        "Income": "Medium",
        "Student": "Yes",
        "Credit_rating": "Fair"
    }

    print(f"Target Test Sample features: {test_sample}")
    prediction, scores, steps = nb_classifier.predict_single(test_sample)
    
    print("\nPosterior Multiplication Steps:")
    for step in steps:
        print(step)
        
    print(f"\n🏆 Final Classification Decision Result: ** {prediction} **")
    print("="*80 + "\n")


📐 STANDARD NAIVE BAYES MATH BREAKDOWN (NO LAPLACE SMOOTHING)

[Step 1: Prior Probabilities P(C)]
  • P(class = 'No') = 5 / 14 = 0.3571
  • P(class = 'Yes') = 9 / 14 = 0.6429

[Step 2: Raw Conditional Probabilities P(Feature = Value | Class)]

  📊 Feature Column: 'Age'
    - P(Age='Y' | class='No') = 3 / 5 = 0.6000
    - P(Age='M' | class='No') = 0 / 5 = 0.0000
    - P(Age='S' | class='No') = 2 / 5 = 0.4000
    - P(Age='Y' | class='Yes') = 2 / 9 = 0.2222
    - P(Age='M' | class='Yes') = 4 / 9 = 0.4444
    - P(Age='S' | class='Yes') = 3 / 9 = 0.3333

  📊 Feature Column: 'Income'
    - P(Income='High' | class='No') = 2 / 5 = 0.4000
    - P(Income='Medium' | class='No') = 2 / 5 = 0.4000
    - P(Income='Low' | class='No') = 1 / 5 = 0.2000
    - P(Income='High' | class='Yes') = 2 / 9 = 0.2222
    - P(Income='Medium' | class='Yes') = 4 / 9 = 0.4444
    - P(Income='Low' | class='Yes') = 3 / 9 = 0.3333

  📊 Feature Column: 'Student'
    - P(Student='No' | class='No') = 4 / 5 = 0.8000
    - P(S